In [126]:

from typing import TypedDict, Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, BaseMessage, AIMessage
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

True

In [127]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [128]:
llm = ChatGroq(model="qwen/qwen3-32b", temperature=0.4, reasoning_format="hidden")

In [129]:
def get_conditional_node(state: ChatState) -> str:
    last_message = state["messages"][-1]
    if isinstance(last_message, HumanMessage) and last_message.content.lower().strip() in ["bye", "exit", "quit"]:
        return "end_chat"
    return "continue_chat"

In [130]:
def ask_question(state: ChatState) -> ChatState:
    input_question = input("Type Bye to exit: ")
    human_message = HumanMessage(content=input_question)
    print(f"Human: {human_message.content}")
    return {"messages": [human_message.content]}

In [131]:
def chat_node(state: ChatState) -> ChatState:
    response = llm.invoke(input=state["messages"])
    print(f"AI: {response.content} \n")
    return {"messages": [AIMessage(content=response.content)]}

In [132]:
checkpointer = InMemorySaver()
graph = StateGraph(ChatState)

graph.add_node("ask_question", ask_question)
graph.add_node("chat_node", chat_node)

graph.add_edge(START, "ask_question")
graph.add_conditional_edges("ask_question", get_conditional_node, {"continue_chat": "chat_node", "end_chat": END})
graph.add_edge("chat_node", "ask_question")

workflow = graph.compile(checkpointer= checkpointer)


In [133]:
# def pretty_print_last_conversation(state: ChatState, reponse: AIMessage):
#     print(f"Human: {state['messages'][-1].content}")
#     print(f"AI: {reponse.content} \n")


In [134]:
# def pretty_print_entire_chat(final_state: ChatState):
#     for message in final_state["messages"]:
#         if isinstance(message, HumanMessage):
#             print(f"Human: {message.content}")
#         elif isinstance(message, AIMessage):
#             print(f"AI: {message.content} \n")


In [135]:
config= {"configurable": {"thread_id": "thread_4"}}

initial_state = {"messages": []}
workflow.invoke(input= initial_state, config= config)


Human: Hello
AI: Hello! How can I assist you today? 

Human: What is 10+32

AI: The sum of 10 and 32 is **42**. 

Fun fact: 42 is also the "Answer to the Ultimate Question of Life, the Universe, and Everything" according to *The Hitchhiker's Guide to the Galaxy*! 😊 

Human: Add 200 to it 
AI: Adding 200 to 42 gives **242**. 

(And just to confirm: 42 + 200 = 242. No need to panic—it’s just a number! 😄) 



KeyboardInterrupt: Interrupted by user

In [ ]:
print("=== State History ===")
for snapshot in workflow.get_state_history(config):
    print(f"--- Step {snapshot.metadata['step']} | {snapshot.created_at} ---")
    print("Next:  ", snapshot.next)
    print("Values:", {k: v[:60] + "..." if isinstance(v, str) and len(v) > 60 else v
                     for k, v in snapshot.values.items()})
    print()

